# Notebook 5: Experiment 1 — Same-Stock Prediction (70/30)## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction**Experiment:** Train on each stock's daily data and predict its own future prices.  **Train/Test Split:** 70/30 (chronological)  **Models:** BiLSTM, BiGRU, LSTM, GRU  **Stocks:** TLKM, BBCA, ASII, UNVR  **Metrics:** MSE, RMSE, MAE, MAPE, R² Score  

In [ ]:
import sys, osimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport warningswarnings.filterwarnings('ignore')sys.path.insert(0, '.')from stock_prediction_utils import *set_seed()set_ieee_style()DATA_DIR = '.'TRAIN_RATIO = 0.7RATIO_LABEL = '70_30'EXP_LABEL = f'Exp1_{RATIO_LABEL}'os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)os.makedirs('results', exist_ok=True)print(f"Experiment 1 - Same Stock Prediction (70/30)")print(f"Train ratio: {TRAIN_RATIO}, Test ratio: {1-TRAIN_RATIO}")

In [ ]:
# Load all daily dataprint("Loading daily data...")daily_data = load_all_daily_data(DATA_DIR)print("\nAll daily data loaded!")

## Run All Experiments

In [ ]:
# ============================================================# EXPERIMENT 1: Train and predict on same stock# ============================================================all_results = []all_predictions = {}  # {stock: {model_type: (y_true, y_pred, dates)}}all_histories = {}    # {stock: {model_type: history}}for stock in STOCKS:    print(f"\n############################################################")    print(f"# STOCK: {stock}")    print(f"############################################################")        # Prepare data    X_train, y_train, X_test, y_test, test_dates = prepare_same_stock_data(        daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK    )    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")        all_predictions[stock] = {}    all_histories[stock] = {}        for model_type in MODEL_TYPES:        exp_name = f'{EXP_LABEL}_{stock}'                y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(            model_type=model_type,            X_train=X_train, y_train=y_train,            X_test=X_test, y_test=y_test,            experiment_name=exp_name,            save_dir=f'models/{EXP_LABEL}',            epochs=EPOCHS, batch_size=BATCH_SIZE        )                # Store results        result = {'Stock': stock, 'Model': model_type, **metrics}        all_results.append(result)        all_predictions[stock][model_type] = (y_true_inv, y_pred_inv, test_dates)        all_histories[stock][model_type] = history                # Plot individual prediction        plot_actual_vs_predicted(            test_dates, y_true_inv, y_pred_inv,            model_type, stock, EXP_LABEL,            save_dir=f'figures/{EXP_LABEL}'        )                # Plot training history        plot_training_history(            history, model_type, stock, EXP_LABEL,            save_dir=f'figures/{EXP_LABEL}'        )print("\n\nAll Experiment 1 (70/30) training complete!")

## Results Summary

In [ ]:
# ============================================================# RESULTS TABLE# ============================================================results_df = pd.DataFrame(all_results)print_results_table(results_df, f"Experiment 1 - Same Stock Prediction (70/30)")# Save resultsresults_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)print(f"Results saved to results/{EXP_LABEL}_results.csv")

## Visualizations

In [ ]:
# ============================================================# ALL MODELS COMPARISON PER STOCK# ============================================================for stock in STOCKS:    y_true = all_predictions[stock][MODEL_TYPES[0]][0]    dates = all_predictions[stock][MODEL_TYPES[0]][2]    preds = {mt: all_predictions[stock][mt][1] for mt in MODEL_TYPES}        plot_all_models_comparison(        dates, y_true, preds, stock, EXP_LABEL,        save_dir=f'figures/{EXP_LABEL}'    )print("All comparison plots saved!")

In [ ]:
# ============================================================# METRICS BAR CHARTS# ============================================================for metric in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2']:    plot_metrics_comparison_bar(        results_df, metric, EXP_LABEL,        group_col='Stock', save_dir=f'figures/{EXP_LABEL}'    )print("All metrics bar charts saved!")

In [ ]:
# ============================================================# SUMMARY: BEST MODEL PER STOCK# ============================================================print("\n" + "="*60)print("  BEST MODEL PER STOCK (by RMSE)")print("="*60)for stock in STOCKS:    stock_results = results_df[results_df['Stock'] == stock]    best_idx = stock_results['RMSE'].idxmin()    best = stock_results.loc[best_idx]    print(f"  {stock}: {best['Model']} (RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")print("\n  BEST MODEL PER STOCK (by R² Score)")print("="*60)for stock in STOCKS:    stock_results = results_df[results_df['Stock'] == stock]    best_idx = stock_results['R2'].idxmax()    best = stock_results.loc[best_idx]    print(f"  {stock}: {best['Model']} (R²={best['R2']:.6f}, RMSE={best['RMSE']:.4f})")